In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np

data = pd.read_csv("weather_data.csv")

print(data.columns)# Convert DATE to actual datetime
data['DATE'] = pd.to_datetime(data['DATE'])

# Sort chronologically
data = data.sort_values('DATE')

# Remove columns we don't need
data = data.drop(columns=['STATION', 'NAME', 'SNWD'])

print(data.head())



In [ ]:
data['TMAX'] = (data['TMAX'] - 32) * 5/9
data['TMIN'] = (data['TMIN'] - 32) * 5/9
print(data.head())
print(data.describe())

In [ ]:
features = data[['PRCP', 'TMAX', 'TMIN']].values
X = []
y = []

sequence_length = 14

for i in range(len(features) - sequence_length):

    X.append(
        features[i:i + sequence_length]
    )

    y.append(
        features[i + sequence_length, 1]
    )
print(X[0])
print(y[0])

X = np.array(X)
y = np.array(y)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

y = y.reshape(-1, 1)

print(X.shape)
print(y.shape)
train_size = int(0.70 * len(X))

val_size = int(0.15 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

In [ ]:
print("Training:")
print(X_train.shape)
print(y_train.shape)

print("Validation:")
print(X_val.shape)
print(y_val.shape)

print("Testing:")
print(X_test.shape)
print(y_test.shape)

In [141]:
X_mean = X_train.mean(dim=(0, 1), keepdim=True)
X_std = X_train.std(dim=(0, 1), keepdim=True)

X_train = (X_train - X_mean) / X_std
X_val = (X_val - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

y_mean = y_train.mean()
y_std = y_train.std()

y_train = (y_train - y_mean) / y_std
y_val = (y_val - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

In [142]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

trainloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

valloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

testloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)


In [ ]:
import torch
import torch.nn as nn

class WeatherModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(42, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: [batch_size, 14, 3]

        x = x.reshape(x.size(0), -1)

        # now x shape: [batch_size, 42]

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))

        x = self.fc3(x)

        return x
model = WeatherModel()   
print(model)

In [ ]:
model = WeatherModel()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 40

for epoch in range(epochs):

    # ----------------------
    # Training
    # ----------------------
    model.train()

    train_loss = 0.0

    for X_batch, y_batch in trainloader:

        predictions = model(X_batch)

        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(trainloader)

    # ----------------------
    # Validation
    # ----------------------
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in valloader:

            predictions = model(X_batch)

            loss = criterion(predictions, y_batch)

            val_loss += loss.item()

    val_loss = val_loss / len(valloader)

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Train Loss: {train_loss:.4f} "
        f"Validation Loss: {val_loss:.4f}"
    )

In [ ]:
model.eval()

test_loss = 0.0
predictions_list = []
actual_list = []

with torch.no_grad():

    for X_batch, y_batch in testloader:

        predictions = model(X_batch)

        loss = criterion(predictions, y_batch)
        test_loss += loss.item()

        predictions_list.append(predictions)
        actual_list.append(y_batch)

test_loss = test_loss / len(testloader)

print("Test Loss:", test_loss)

predictions = torch.cat(predictions_list)
actual = torch.cat(actual_list)
predictions_real = predictions * y_std + y_mean
actual_real = actual * y_std + y_mean

mae = torch.mean(torch.abs(predictions_real - actual_real))

rmse = torch.sqrt(
    torch.mean((predictions_real - actual_real) ** 2)
)

print("MAE:", mae.item())
print("RMSE:", rmse.item())
